## Problema 2 - Optimización con multiplicadores de Lagrange y Newton-Raphson multidimensional
### Consideraciones para la Resolución

* **Utilizar seis decimales para los resultados.**
* **Utilizar todos los decimales para los calculos.**
* **No utilizar formato exponencial (notacion científica).**
* **Utilizar una tolerancia de $10^{-3}$.**
* **No modificar los valores iniciales ni la tolerancia para los calculos.**
* **No es necesario realizar ninguna conversion de unidades.**

Se quiere construir un silo (incluir la base circular). Tiene forma de cilindro rematado por una semiesfera en el techo. El costo de construcción por unidad cuadrada del área superficial es de 24 dolares para la semiesfera, para la pared cilíndrica de 14 dolares y la base de 35 dolares por cada metro cuadrado. Determinar las dimensiones (radio de semiesfera y altura del cilindro) que se deben utilizar si se quiere minimizar los costos y el volumen fijo se pretende sea de 1000 metros cúbicos.
Utilizar el método de Newton directo con ayuda de un multiplicador de Lagrange de modo a minimizar el gasto. Utilizar como valor inicial al valor de 10 de todas las variables y una tolerancia de $10^{-3}$.

Determinar:
- Item a: Dimensiones del radio
- Item b: Dimensiones de la altura
- Item c: Valor absoluto del multiplicador de Lagrange
- Item d: Costo del Silo


_Planteamiento_

Para las Áreas de las componentes del silo:
$$
\begin{aligned}
A_p &= 2\pi r h \quad \text{(pared cilíndrica)} \\
A_t &= 2\pi r^2 \quad \text{(techo semiesférico)} \\
A_b &= \pi r^2 \quad \text{(base circular)}
\end{aligned}
$$

El costo total está dado por la suma de los costos de cada parte:
$$
\begin{aligned}
C(A_p, A_t, A_b) &= 14A_p + 24A_t + 35A_b \\
C(r, h) &= 14(2\pi r h) + 24(2\pi r^2) + 35(\pi r^2) \\
C(r, h) &= 28\pi r h + 83\pi r^2
\end{aligned}
$$

El volumen total se compone del cilindro y la semiesfera:
$$
\begin{aligned}
V_p &= \pi r^2 h \quad \text{(Interior de la pared cilíndrica)} \\
V_t &= \frac{2}{3}\pi r^3 \quad \text{(Interior del techo semiesférico)} \\
V_T &= \pi r^2 h + \frac{2}{3}\pi r^3 = 1000
\end{aligned}
$$

Definimos la función de Lagrange $L(r, h, \lambda)$ considerando la restricción de volumen:
$$L(r, h, \lambda) = (28\pi r h + 83\pi r^2) + \lambda \left( 1000 - \pi r^2 h - \frac{2}{3}\pi r^3 \right)$$

Para encontrar los puntos críticos, resolvemos:
$$
\begin{aligned}
\frac{\partial L}{\partial r} &= 0 \\
\frac{\partial L}{\partial h} &= 0 \\
\frac{\partial L}{\partial \lambda} &= 0
\end{aligned}
$$
Es decir $\nabla L = 0$ siendo $\nabla = \left( \frac{\partial }{\partial r}, \frac{\partial }{\partial h}, \frac{\partial }{\partial \lambda} \right)$.

In [ ]:
import jax.numpy as jnp
from jax import grad
from jax import jacfwd

# Costo del silo
def C(r,h):
    return 28*jnp.pi*r*h + 83*jnp.pi*jnp.pow(r,2)

# Volumen del silo
def V(r,h):
    return jnp.pi*jnp.pow(r,2)*h + 2/3*jnp.pi*jnp.pow(r,3)

# Multiplicador de Lagrange
def L(var):
    r,h,lm = var
    return C(r,h) + lm*(1000 - V(r,h))

# Sistema de ecuaciones no lineales
sistema = grad(L)

# PARAMETROS DE INICIALIZACION
P0 = jnp.array([10, 10, 10], dtype=float)
tol = 1e-3
m = 50
jacob_sist = jacfwd(sistema)

################RUTINA009##########################
for i in range(m):
    F = sistema(P0)
    J = jacob_sist(P0)
    deltaP = jnp.linalg.solve(J, -F)
    P = P0 + deltaP
    err = jnp.linalg.norm(deltaP)
    relerr = err / jnp.linalg.norm(P)
    f_norm = jnp.linalg.norm(F)
    if (err < tol) or (relerr < tol) or (f_norm < tol):
        P0 = P
        break
    P0 = P

print("Analsis de convergencia:")
print("Error absoluto: ", err)
print("Error relativo: ", relerr)
print("Cantidad de iteraciones: ", i+1)
print("Valor de |F(P)|: ", f_norm)

dim_r = P[0]
dim_h = P[1]
L_res = jnp.abs(L(P))
C_res = C(dim_r,dim_h)

print("------------------")
print("a) Radio del silo: ", jnp.around(dim_r,6))
print("b) Altura del silo: ", jnp.around(dim_h,6))
print("c) Valor del multiplicador de Lagrange: ", jnp.around(L_res,6))
print("d) Costo del silo: ", jnp.around(C_res,6))


Analsis de convergencia:
Error absoluto:  0.0014573977
Error relativo:  8.100869e-05
Cantidad de iteraciones:  6
Valor de |F(P)|:  0.12638995
------------------
a) Radio del silo:  4.106899
b) Altura del silo:  16.134247
c) Valor del multiplicador de Lagrange:  10226.693
d) Costo del silo:  10226.693
